# Phase 4: ML Classification

Train and evaluate ML classifiers on the extracted features using Leave One Subject Out Cross-Validation (LOSO-CV).

**Classifiers**:
1. Support Vector Machine (SVM) [strong for small/medium datasets]
2. Random Forest  [handles many features well, gives feature importance]
3. K-Nearest Neighbors (KNN)  [simple baseline]

**Evaluation**:
- LOSO-CV: Train on 39 subjects, test on 1, repeat for all 40. This tests if the model can generalize to a completely new person it has never seen.
- Metrics: Accuracy, Precision, Recall, F1-Score, Confusion Matrix

**Classification Tasks**:
- Binary: Stress (Low + High) vs Relaxed
- 3 class: Relaxed vs Low Stress vs High Stress

In [1]:

# Imports:

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report)
import time
import warnings
warnings.filterwarnings('ignore')
 
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})

# Configuration

In [2]:

BASE_DIR = os.path.join("..", "")
 
DATA_DIR = os.path.join(BASE_DIR, "Results", "phase3")
OUTPUT_DIR = os.path.join(BASE_DIR, "Results", "phase4")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# LOSO CV Engine

In a regular train/test split, data from the SAME person can end up in both training and testing sets. This is a problem because:
- EEG signals from the same person are correlated (similar brain anatomy, similar baseline rhythms)
- The model might learn '*this is Subject 5*' instead of '*this is stress*'
- Results look great in the lab but fail on new users

LOSO-CV prevents this by ensuring all data from one subject is held out for testing, and the model is trained on the remaining 39 subjects. We repeat this 40 times (once per subject) and average the results.

In [3]:

"""
Parameters:
    X: feature matrix (n_samples, n_features)
    y: labels (n_samples,)
    subjects: subject IDs (n_samples,)
    classifier_fn: function that returns a fresh classifier instance
    classifier_name: string name for printing
    
Returns:
    results dict with predictions, per-subject accuracy, overall metrics
"""

def loso_cv(X, y, subjects, classifier_fn, classifier_name="Classifier"):

    unique_subjects = sorted(np.unique(subjects))
    n_subjects = len(unique_subjects)
    
    all_true = []
    all_pred = []
    subject_accuracies = {}
    
    print(f"\n  Running LOSO-CV with {classifier_name}...")
    start_time = time.time()
    
    for i, test_subject in enumerate(unique_subjects):
        # Split: test = this subject, train = all others
        test_mask = subjects == test_subject
        train_mask = ~test_mask
        
        X_train, X_test = X[train_mask], X[test_mask]
        y_train, y_test = y[train_mask], y[test_mask]
        
        # Standardize features (fit on training data only)
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)
        
        # Train (fitting) and predicting
        clf = classifier_fn()
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        
        # Store results
        all_true.extend(y_test)
        all_pred.extend(y_pred)
        
        acc = accuracy_score(y_test, y_pred)
        subject_accuracies[test_subject] = acc
        
        # Progress
        if (i + 1) % 10 == 0 or i == 0:
            elapsed = time.time() - start_time
            remaining = elapsed / (i + 1) * (n_subjects - i - 1)
            print(f"    Subject {i+1}/{n_subjects} "
                  f"(acc: {acc:.1%}) "
                  f"[{elapsed:.0f}s elapsed, ~{remaining:.0f}s remaining]")
    
    total_time = time.time() - start_time
    all_true = np.array(all_true)
    all_pred = np.array(all_pred)
    
    # Overall metrics
    overall_acc = accuracy_score(all_true, all_pred)
    
    # Handling multi-class vs binary for averaging
    unique_labels = np.unique(all_true)
    avg_method = 'binary' if len(unique_labels) == 2 else 'weighted'
    pos_label = unique_labels[1] if len(unique_labels) == 2 else None
    
    precision = precision_score(all_true, all_pred, average=avg_method,
                                pos_label=pos_label, zero_division=0)
    recall = recall_score(all_true, all_pred, average=avg_method,
                          pos_label=pos_label, zero_division=0)
    f1 = f1_score(all_true, all_pred, average=avg_method,
                  pos_label=pos_label, zero_division=0)
    
    cm = confusion_matrix(all_true, all_pred)
    
    results = {
        'classifier': classifier_name,
        'accuracy': overall_acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'confusion_matrix': cm,
        'all_true': all_true,
        'all_pred': all_pred,
        'subject_accuracies': subject_accuracies,
        'time': total_time,
        'unique_labels': unique_labels,
    }
    
    print(f"    Done in {total_time:.1f}s — Overall accuracy: {overall_acc:.1%}")
    
    return results

# Classifiers

- **SVM**: Works well on high dimensional data (480 features). The RBF kernel can capture non-linear relationships. Published SAM 40 research uses SVM as the primary classifier.
    
- **Random Forest**: Ensemble of decision trees. Naturally handles many features, provides built in feature importance rankings, and is robust to outliers. Good baseline.
    
- **KNN**: Simple but effective. Classifies based on the closest training examples.

In [4]:

"""
Returns:
        dict of {name: classifier_factory_function}
"""

def get_classifiers():
    return {
        'SVM (RBF)': lambda: SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
        'Random Forest': lambda: RandomForestClassifier(
            n_estimators=100, max_depth=None, random_state=42, n_jobs=-1),
        'KNN (k=5)': lambda: KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    }

# Visualization

Plot confusion matrices for all classifiers side by side.

In [17]:

def plot_confusion_matrices(all_results, class_names, task_label, output_dir):

    n_clf = len(all_results)
    fig, axes = plt.subplots(1, n_clf, figsize=(6 * n_clf, 5))
    if n_clf == 1:
        axes = [axes]
    
    fig.suptitle(f'Confusion Matrices — {task_label}\n'
                 f'LOSO-CV (Leave-One-Subject-Out)',
                 fontsize=14, fontweight='bold')
    
    for idx, (results, ax) in enumerate(zip(all_results, axes)):
        cm = results['confusion_matrix']
        # Normalize by row (true labels) to show percentages
        cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
        
        sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='Blues',
                    xticklabels=class_names, yticklabels=class_names,
                    ax=ax, cbar_kws={'label': '%'}, vmin=0, vmax=100)
        
        # Also show raw counts
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j + 0.5, i + 0.75, f'(n={cm[i,j]})',
                        ha='center', va='center', fontsize=7, color='gray')
        
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.set_title(f'{results["classifier"]}\n'
                     f'Acc: {results["accuracy"]:.1%} | F1: {results["f1"]:.1%}',
                     fontweight='bold', fontsize=11)
    
    plt.tight_layout()
    filename = f'01_confusion_matrix_{task_label.lower().replace(" ", "_")}.png'
    plt.savefig(os.path.join(output_dir, filename), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved: {filename}")

Show how accuracy varies across subjects for each classifier

In [18]:
def plot_per_subject_accuracy(all_results, task_label, output_dir):

    fig, axes = plt.subplots(len(all_results), 1,
                             figsize=(14, 4 * len(all_results)), sharex=True)
    if len(all_results) == 1:
        axes = [axes]
    
    fig.suptitle(f'Per-Subject Accuracy — {task_label}\n'
                 f'Each bar = accuracy when that subject is the test set',
                 fontsize=14, fontweight='bold')
    
    colors_clf = ['#3498db', '#2ecc71', '#e74c3c']
    
    for idx, (results, ax) in enumerate(zip(all_results, axes)):
        subj_acc = results['subject_accuracies']
        subjects = sorted(subj_acc.keys())
        accs = [subj_acc[s] for s in subjects]
        
        bars = ax.bar(range(len(subjects)), accs, color=colors_clf[idx], alpha=0.7)
        ax.axhline(results['accuracy'], color='black', linestyle='--', alpha=0.5,
                   label=f'Mean: {results["accuracy"]:.1%}')
        ax.set_ylabel('Accuracy')
        ax.set_title(f'{results["classifier"]}', fontweight='bold')
        ax.legend(fontsize=9)
        ax.set_ylim(0, 1.05)
        
        # Color low accuracy subjects differently
        mean_acc = results['accuracy']
        for bar, acc in zip(bars, accs):
            if acc < mean_acc - 0.2:
                bar.set_color('#e74c3c')
                bar.set_alpha(0.9)
    
    axes[-1].set_xlabel('Subject ID')
    axes[-1].set_xticks(range(len(subjects)))
    axes[-1].set_xticklabels(subjects, fontsize=7, rotation=0)
    
    plt.tight_layout()
    filename = f'02_per_subject_accuracy_{task_label.lower().replace(" ", "_")}.png'
    plt.savefig(os.path.join(output_dir, filename), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved: {filename}")

Summary comparison chart of all classifiers across both tasks.

In [19]:
def plot_results_comparison(binary_results, multi_results, output_dir):

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Model Comparison Summary — LOSO-CV Results',
                 fontsize=14, fontweight='bold')
    
    metrics = ['accuracy', 'precision', 'recall', 'f1']
    metric_labels = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    colors_clf = ['#3498db', '#2ecc71', '#e74c3c']
    
    for ax, results_list, title in [(axes[0], binary_results, 'Binary (Stress vs Relaxed)'),
                                     (axes[1], multi_results, '3-Class (R / L / H)')]:
        x = np.arange(len(metrics))
        width = 0.25
        
        for i, results in enumerate(results_list):
            values = [results[m] for m in metrics]
            ax.bar(x + i * width, values, width, label=results['classifier'],
                   color=colors_clf[i], alpha=0.8, edgecolor='white')
        
        ax.set_xticks(x + width)
        ax.set_xticklabels(metric_labels)
        ax.set_ylabel('Score')
        ax.set_title(title, fontweight='bold')
        ax.legend(fontsize=8)
        ax.set_ylim(0, 1.05)
        
        # Add value labels on bars
        for i, results in enumerate(results_list):
            values = [results[m] for m in metrics]
            for j, v in enumerate(values):
                ax.text(x[j] + i * width, v + 0.01, f'{v:.2f}',
                        ha='center', fontsize=7, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, '03_model_comparison.png'), dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved: 03_model_comparison.png")

Saving a clean results table as CSV.

In [9]:
def save_results_table(binary_results, multi_results, output_dir):
    
    rows = []
    for task_name, results_list in [('Binary', binary_results), ('3-Class', multi_results)]:
        for r in results_list:
            rows.append({
                'Task': task_name,
                'Classifier': r['classifier'],
                'Accuracy': f"{r['accuracy']:.4f}",
                'Precision': f"{r['precision']:.4f}",
                'Recall': f"{r['recall']:.4f}",
                'F1-Score': f"{r['f1']:.4f}",
                'Time (s)': f"{r['time']:.1f}",
            })
    
    results_df = pd.DataFrame(rows)
    results_df.to_csv(os.path.join(output_dir, 'classification_results.csv'), index=False)
    
    print(f"\n  Results Table:")
    print(results_df.to_string(index=False))
    return results_df

# MAIN

In [10]:
if __name__ == "__main__":
    print("=" * 60)
    print("  EEG COGNITIVE STRESS CLASSIFICATION")
    print("  Phase 4: Classification (LOSO-CV)")
    print("=" * 60)

  EEG COGNITIVE STRESS CLASSIFICATION
  Phase 4: Classification (LOSO-CV)


In [11]:
# Loading features from Phase 3

print(f"\n▶ Loading features from Phase 3...")
    
features_path = os.path.join(DATA_DIR, 'features.npz')
labels_path = os.path.join(DATA_DIR, 'features_with_labels.csv')
    
if not os.path.exists(features_path):
    print(f"  ✗ Not found: {features_path}")
    print(f"  Run Phase 3 first.")
    exit(1)
    
data = np.load(features_path, allow_pickle=True)
X = data['feature_matrix']
feature_names = list(data['feature_names'])
    
labels_df = pd.read_csv(labels_path)
subjects = labels_df['subject'].values
    
print(f"  Feature matrix: {X.shape}")
print(f"  Subjects: {len(np.unique(subjects))}")


▶ Loading features from Phase 3...
  Feature matrix: (2400, 480)
  Subjects: 40


## Binary Classification

Stress vs Relax Classification

In [20]:
# Create binary labels: Relaxed=0, Stress=1 (Low + High combined)

y_binary = np.array([
    0 if level == 'Relaxed' else 1 
    for level in labels_df['stress_level']
])
    
n_relaxed = (y_binary == 0).sum()
n_stress = (y_binary == 1).sum()
print(f"  Relaxed: {n_relaxed} | Stress: {n_stress}")
    
classifiers = get_classifiers()
binary_results = []
    
for clf_name, clf_fn in classifiers.items():
    results = loso_cv(X, y_binary, subjects, clf_fn, clf_name)
    binary_results.append(results)
    
# Visualize binary results
print(f"\n▶ Generating binary classification plots...")
plot_confusion_matrices(binary_results, ['Relaxed', 'Stress'], 'Binary', OUTPUT_DIR)
plot_per_subject_accuracy(binary_results, 'Binary', OUTPUT_DIR)

  Relaxed: 1110 | Stress: 1290

  Running LOSO-CV with SVM (RBF)...
    Subject 1/40 (acc: 61.7%) [1s elapsed, ~48s remaining]
    Subject 10/40 (acc: 75.0%) [18s elapsed, ~54s remaining]
    Subject 20/40 (acc: 38.3%) [28s elapsed, ~28s remaining]
    Subject 30/40 (acc: 71.7%) [37s elapsed, ~12s remaining]
    Subject 40/40 (acc: 71.7%) [47s elapsed, ~0s remaining]
    Done in 47.1s — Overall accuracy: 57.9%

  Running LOSO-CV with Random Forest...
    Subject 1/40 (acc: 66.7%) [1s elapsed, ~39s remaining]
    Subject 10/40 (acc: 71.7%) [10s elapsed, ~29s remaining]
    Subject 20/40 (acc: 50.0%) [20s elapsed, ~20s remaining]
    Subject 30/40 (acc: 58.3%) [29s elapsed, ~10s remaining]
    Subject 40/40 (acc: 61.7%) [39s elapsed, ~0s remaining]
    Done in 39.0s — Overall accuracy: 58.7%

  Running LOSO-CV with KNN (k=5)...
    Subject 1/40 (acc: 48.3%) [0s elapsed, ~1s remaining]
    Subject 10/40 (acc: 45.0%) [0s elapsed, ~1s remaining]
    Subject 20/40 (acc: 50.0%) [1s elapsed, ~

## 3 Class Classification

Relaxed vs Low vs High Classification

In [21]:

# Create 3 class labels: Relaxed=0, Low Stress=1, High Stress=2
label_map = {'Relaxed': 0, 'Low Stress': 1, 'High Stress': 2}
y_multi = np.array([label_map[level] for level in labels_df['stress_level']])
    
for label, code in label_map.items():
    print(f"  {label}: {(y_multi == code).sum()}")
    
multi_results = []
    
for clf_name, clf_fn in classifiers.items():
    results = loso_cv(X, y_multi, subjects, clf_fn, clf_name)
    multi_results.append(results)
    
# Visualize 3-class results
print(f"\n▶ Generating 3-class classification plots...")
plot_confusion_matrices(multi_results, ['Relaxed', 'Low', 'High'], '3-Class', OUTPUT_DIR)
plot_per_subject_accuracy(multi_results, '3-Class', OUTPUT_DIR)

  Relaxed: 1110
  Low Stress: 940
  High Stress: 350

  Running LOSO-CV with SVM (RBF)...
    Subject 1/40 (acc: 48.3%) [1s elapsed, ~50s remaining]
    Subject 10/40 (acc: 70.0%) [11s elapsed, ~33s remaining]
    Subject 20/40 (acc: 21.7%) [22s elapsed, ~22s remaining]
    Subject 30/40 (acc: 48.3%) [33s elapsed, ~11s remaining]
    Subject 40/40 (acc: 61.7%) [44s elapsed, ~0s remaining]
    Done in 43.6s — Overall accuracy: 47.5%

  Running LOSO-CV with Random Forest...
    Subject 1/40 (acc: 46.7%) [1s elapsed, ~42s remaining]
    Subject 10/40 (acc: 58.3%) [10s elapsed, ~31s remaining]
    Subject 20/40 (acc: 26.7%) [21s elapsed, ~21s remaining]
    Subject 30/40 (acc: 35.0%) [31s elapsed, ~10s remaining]
    Subject 40/40 (acc: 61.7%) [42s elapsed, ~0s remaining]
    Done in 41.9s — Overall accuracy: 47.4%

  Running LOSO-CV with KNN (k=5)...
    Subject 1/40 (acc: 31.7%) [0s elapsed, ~1s remaining]
    Subject 10/40 (acc: 46.7%) [0s elapsed, ~1s remaining]
    Subject 20/40 (acc:

# Summary

In [22]:

# Combined Comparison:
print(f"\n▶ Generating comparison plots...")
plot_results_comparison(binary_results, multi_results, OUTPUT_DIR)
results_df = save_results_table(binary_results, multi_results, OUTPUT_DIR)

print(f"\n{'='*60}")
print(f"  PHASE 4 COMPLETE!")
print(f"{'='*60}")
    
print(f"\n  BINARY RESULTS (Stress vs Relaxed):")
best_binary = max(binary_results, key=lambda r: r['accuracy'])
for r in binary_results:
    marker = " ★ BEST" if r == best_binary else ""
    print(f"    {r['classifier']:20s}  Acc: {r['accuracy']:.1%}  F1: {r['f1']:.1%}{marker}")
    
print(f"\n  3-CLASS RESULTS (Relaxed / Low / High):")
best_multi = max(multi_results, key=lambda r: r['accuracy'])
for r in multi_results:
    marker = " ★ BEST" if r == best_multi else ""
    print(f"    {r['classifier']:20s}  Acc: {r['accuracy']:.1%}  F1: {r['f1']:.1%}{marker}")
    
print(f"\n  Evaluation method: Leave-One-Subject-Out CV ({len(np.unique(subjects))} folds)")
print(f"\n  Outputs in: {os.path.abspath(OUTPUT_DIR)}/")
print(f"    01_confusion_matrix_binary.png")
print(f"    01_confusion_matrix_3-class.png")
print(f"    02_per_subject_accuracy_binary.png")
print(f"    02_per_subject_accuracy_3-class.png")
print(f"    03_model_comparison.png")
print(f"    classification_results.csv")
print(f"\n  Next → Phase 5: Deep Learning (optional)")
print(f"{'='*60}")



▶ Generating comparison plots...
  ✓ Saved: 03_model_comparison.png

  Results Table:
   Task    Classifier Accuracy Precision Recall F1-Score Time (s)
 Binary     SVM (RBF)   0.5787    0.5983 0.6581   0.6268     47.1
 Binary Random Forest   0.5867    0.6066 0.6574   0.6310     39.0
 Binary     KNN (k=5)   0.5221    0.5522 0.5860   0.5686      1.5
3-Class     SVM (RBF)   0.4750    0.4142 0.4750   0.4365     43.6
3-Class Random Forest   0.4738    0.4555 0.4738   0.4344     41.9
3-Class     KNN (k=5)   0.4321    0.3897 0.4321   0.4057      1.5

  PHASE 4 COMPLETE!

  BINARY RESULTS (Stress vs Relaxed):
    SVM (RBF)             Acc: 57.9%  F1: 62.7%
    Random Forest         Acc: 58.7%  F1: 63.1% ★ BEST
    KNN (k=5)             Acc: 52.2%  F1: 56.9%

  3-CLASS RESULTS (Relaxed / Low / High):
    SVM (RBF)             Acc: 47.5%  F1: 43.7% ★ BEST
    Random Forest         Acc: 47.4%  F1: 43.4%
    KNN (k=5)             Acc: 43.2%  F1: 40.6%

  Evaluation method: Leave-One-Subject-Out CV